# Module 1: Telco Customer Churn Prediction & Decision Modeling

Welcome to **Module 1**! In this practical lab, you will analyze customer churn for a telecommunications provider (`Telco-Customer-Churn.csv` — 7,043 customers, 21 columns).

### Business Brief:
A telecom company loses ~25% of its customers annually. Winning new customers costs 5x more than retaining existing ones. Your brief:
> *"Predict which customers are likely to churn, identify root causes, and determine the optimal intervention cutoff for retention calls."*

---

### Assignment Tasks Overview:
1. **Task 1**: Load `Telco-Customer-Churn.csv`, inspect schema, convert `TotalCharges` strings to float, and impute missing values.
2. **Task 2**: Exploratory Data Analysis — Analyze churn rates across contract types, tenure buckets, and monthly charges.
3. **Task 3**: Preprocessing & Feature Encoding — One-hot encode categorical features and scale numeric variables.
4. **Task 4**: Machine Learning Model Training — Train Logistic Regression and Random Forest classifiers.
5. **Task 5**: Business Decision Threshold Modeling — Evaluate Precision-Recall trade-offs to maximize retention campaign ROI.

In [17]:
# Task 1: Load Telco Churn Dataset & Handle Data Types
import pandas as pd
import numpy as np

# Load dataset (pre-mounted in workspace)
df = pd.read_csv('Telco-Customer-Churn.csv')

# Clean TotalCharges (spaces/blank strings -> NaN -> float)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Encode target variable
df['Churn_Numeric'] = (df['Churn'] == 'Yes').astype(int)

print(f"Dataset Shape: {df.shape}")
print("Churn Distribution:\n", df['Churn'].value_counts(normalize=True))
df.head()

# Check dataset structure
print("Dataset shape:", df.shape)

print("\nColumn data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTotalCharges summary:")
print(df["TotalCharges"].describe())

print("\nChurn counts:")
print(df["Churn"].value_counts())

print("\nChurn percentages:")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

print("\nFirst five rows:")
df.head()


Dataset Shape: (7043, 22)
Churn Distribution:
 No     0.73463
Yes    0.26537
Name: Churn, dtype: float64
Dataset shape: (7043, 22)

Column data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
Churn_Numeric         int32
dtype: object

Missing values:
Series([], dtype: int64)

Duplicate rows: 0

TotalCharges summary:
count    7043.000000
mean     2281.916928
std      2265.270398
min        18.800000
25%       402.225000
50%      1397.475000
75%      

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>customerID</th>
      <th>gender</th>
      <th>SeniorCitizen</th>
      <th>Partner</th>
      <th>Dependents</th>
      <th>tenure</th>
      <th>PhoneService</th>
      <th>MultipleLines</th>
      <th>InternetService</th>
      <th>OnlineSecurity</th>
      <th>OnlineBackup</th>
      <th>DeviceProtection</th>
      <th>TechSupport</th>
      <th>StreamingTV</th>
      <th>StreamingMovies</th>
      <th>Contract</th>
      <th>PaperlessBilling</th>
      <th>PaymentMethod</th>
      <th>MonthlyCharges</th>
      <th>TotalCharges</th>
      <th>Churn</th>
      <th>Churn_Numeric</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
     

### Task 2: Exploratory Data Analysis & Churn Drivers
Analyze how Contract Type, Tenure, and Internet Service impact churn rates.

In [18]:
# Task 2: Churn Analysis by Contract & Service
contract_churn = df.groupby('Contract')['Churn_Numeric'].agg(['count', 'mean']).rename(columns={'mean': 'Churn_Rate'})
contract_churn['Churn_Rate'] = (contract_churn['Churn_Rate'] * 100).round(2).astype(str) + '%'

print("--- Churn Rate by Contract Type ---")
print(contract_churn)

# Tenure Grouping
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-1 yr', '1-2 yrs', '2-4 yrs', '4+ yrs'])
tenure_churn = df.groupby('TenureGroup')['Churn_Numeric'].mean() * 100
print("\n--- Churn Rate by Tenure Group ---")
print(tenure_churn.round(2).astype(str) + '%')

--- Churn Rate by Contract Type ---
                count Churn_Rate
Contract                        
Month-to-month   3875     42.71%
One year         1473     11.27%
Two year         1695      2.83%

--- Churn Rate by Tenure Group ---
TenureGroup
0-1 yr     47.68%
1-2 yrs    28.71%
2-4 yrs    20.39%
4+ yrs      9.51%
Name: Churn_Numeric, dtype: object



### Task 3: Model Training & Evaluation
Train a Random Forest classifier and generate a Confusion Matrix.

In [19]:
# Task 3: Preprocessing & Baseline Classifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

feature_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X = df[feature_cols]
y = df['Churn_Numeric']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.87      0.84      1035
           1       0.55      0.44      0.48       374

    accuracy                           0.75      1409
   macro avg       0.68      0.65      0.66      1409
weighted avg       0.74      0.75      0.74      1409




## Task 4: Machine Learning Model Training
Train Logistic Regression and Random Forest classifiers to predict customer churn.
Evaluate both models using Accuracy, Precision, Recall, F1-score, and ROC-AUC to identify the best-performing model.

In [24]:

# Task 4: Machine Learning Model Training

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

# 1. Separate features and target
X = df.drop(
    columns=["Churn", "Churn_Numeric", "customerID", "TenureGroup"],
    errors="ignore"
)
y = df["Churn_Numeric"]

# 2. Identify numerical and categorical columns
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

# 3. Preprocess numerical and categorical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# 4. Split into training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 5. Define both machine learning models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42
    )
}

# 6. Train and evaluate both models
results = []
trained_models = {}

for name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

    trained_models[name] = pipeline

    print(f"\n{'=' * 50}")
    print(name)
    print("=" * 50)
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

# 7. Compare model performance
results_df = pd.DataFrame(results)
print("\n--- Model Performance Comparison ---")
print(results_df.round(3).to_string(index=False))

# 8. Identify the model with the highest ROC-AUC
best_model_name = results_df.loc[
    results_df["ROC-AUC"].idxmax(), "Model"
]

print("Highest ROC-AUC model:", best_model_name)



Logistic Regression
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

Confusion Matrix:
[[747 288]
 [ 81 293]]

Random Forest
              precision    recall  f1-score   support

           0       0.82      0.90      0.86      1035
           1       0.62      0.47      0.54       374

    accuracy                           0.78      1409
   macro avg       0.72      0.68      0.70      1409
weighted avg       0.77      0.78      0.77      1409

Confusion Matrix:
[[928 107]
 [197 177]]

--- Model Performance Comparison ---
              Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression     0.738      0.504   0.783     0.614    0.841
      Random Forest     0.784      0.623   0.473     0

## Task 5: Business Decision Threshold Modeling
Evaluate different decision thresholds to balance Precision and Recall and identify a cost-effective strategy for customer retention campaigns.

In [25]:

import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------------------------------------
# TASK 5: Business Decision Threshold Modeling
# --------------------------------------------------

# Illustrative business assumptions — change these
# according to actual company data.
campaign_cost_per_customer = 100       # ₹ per customer contacted
value_per_retained_customer = 5000     # ₹
campaign_success_rate = 0.20           # 20% of true churners contacted are retained

thresholds = np.arange(0.10, 0.91, 0.05)

results = []

for model_name, model in trained_models.items():

    # Predicted probability of churn
    churn_probabilities = model.predict_proba(X_test)[:, 1]

    for threshold in thresholds:

        # Apply the decision threshold
        y_pred_threshold = (
            churn_probabilities >= threshold
        ).astype(int)

        # Calculate evaluation metrics
        precision = precision_score(
            y_test, y_pred_threshold, zero_division=0
        )

        recall = recall_score(
            y_test, y_pred_threshold, zero_division=0
        )

        f1 = f1_score(
            y_test, y_pred_threshold, zero_division=0
        )

        tn, fp, fn, tp = confusion_matrix(
            y_test, y_pred_threshold, labels=[0, 1]
        ).ravel()

        # Estimate campaign economics
        customers_targeted = int(tp + fp)
        expected_retained = tp * campaign_success_rate

        campaign_cost = (
            customers_targeted * campaign_cost_per_customer
        )

        expected_benefit = (
            expected_retained * value_per_retained_customer
        )

        net_benefit = expected_benefit - campaign_cost

        roi_percent = (
            (net_benefit / campaign_cost) * 100
            if campaign_cost > 0 else 0
        )

        results.append({
            "Model": model_name,
            "Threshold": round(float(threshold), 2),
            "Precision": precision,
            "Recall": recall,
            "F1_Score": f1,
            "Customers_Targeted": customers_targeted,
            "True_Churners_Found": int(tp),
            "Missed_Churners": int(fn),
            "Campaign_Cost_Rs": campaign_cost,
            "Expected_Benefit_Rs": expected_benefit,
            "Net_Benefit_Rs": net_benefit,
            "Estimated_ROI_Percent": roi_percent
        })

# Create results table
threshold_results = pd.DataFrame(results)

print("TASK 5: THRESHOLD PERFORMANCE")
print(threshold_results.round(2).to_string(index=False))

# Find the threshold with the highest estimated ROI
best_index = threshold_results["Estimated_ROI_Percent"].idxmax()
best_result = threshold_results.loc[best_index]

print("\n--- BEST ESTIMATED BUSINESS THRESHOLD ---")
print("Model:", best_result["Model"])
print("Threshold:", best_result["Threshold"])
print("Precision:", round(best_result["Precision"], 3))
print("Recall:", round(best_result["Recall"], 3))
print("Customers targeted:", int(best_result["Customers_Targeted"]))
print("Estimated campaign cost: ₹",
      round(best_result["Campaign_Cost_Rs"], 2))
print("Estimated benefit: ₹",
      round(best_result["Expected_Benefit_Rs"], 2))
print("Estimated net benefit: ₹",
      round(best_result["Net_Benefit_Rs"], 2))
print("Estimated ROI (%):",
      round(best_result["Estimated_ROI_Percent"], 2))


TASK 5: THRESHOLD PERFORMANCE
              Model  Threshold  Precision  Recall  F1_Score  Customers_Targeted  True_Churners_Found  Missed_Churners  Campaign_Cost_Rs  Expected_Benefit_Rs  Net_Benefit_Rs  Estimated_ROI_Percent
Logistic Regression       0.10       0.34    0.99      0.51                1083                  370                4            108300             370000.0        261700.0                 241.64
Logistic Regression       0.15       0.37    0.98      0.53                 994                  365                9             99400             365000.0        265600.0                 267.20
Logistic Regression       0.20       0.39    0.96      0.55                 927                  360               14             92700             360000.0        267300.0                 288.35
Logistic Regression       0.25       0.41    0.94      0.57                 863                  351               23             86300             351000.0        264700.0              